# Ablation study — PID-UNet v2

6 cấu hình, cùng loss và split. Chọn GPU trong Runtime → Change runtime type.
Chạy seed 42 trước; sau đó đổi `SEEDS` thành `[42, 3407, 1234]` để đủ 18 runs.
Mọi checkpoint và kết quả được lưu trên Drive. Chạy lại cùng notebook để tiếp tục sau khi ngắt kết nối.
Chỉ chạy một bộ runner trên thư mục này tại một thời điểm. Không đổi cấu hình giữa các runs của cùng study.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, re, subprocess, sys
OUTPUT_ROOT = Path('/content/drive/MyDrive/pid_v2_ablation_v1')
REPO = Path('/content/polyp-segmentation')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/nguyenphucvinh111/polyp-segmentation.git', str(REPO)], check=True)
os.chdir(REPO)
manifest = OUTPUT_ROOT / 'ablation_study.json'
if manifest.exists():
    revision = json.loads(manifest.read_text())['provenance']['git_revision']
    assert re.fullmatch(r'[0-9a-f]{40}', revision), 'Missing valid source revision'
    subprocess.run(['git', 'fetch', 'origin'], check=True)
    subprocess.run(['git', 'checkout', revision], check=True)
else:
    subprocess.run(['git', 'checkout', 'main'], check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before training'
print('GPU:', torch.cuda.get_device_name(0))
print('Persistent outputs:', OUTPUT_ROOT)


In [ ]:
VARIANTS = ['full', 'no_global', 'no_derivative', 'fixed_gains', 'no_boundary_gate', 'no_gate_floor']
SEEDS = [42]  # Then use [42, 3407, 1234]; completed runs are skipped.
command = [sys.executable, '-u', 'run_pid_v2_ablation.py',
           '--output_root', str(OUTPUT_ROOT), '--data_root', '/content/polyp-segmentation/data',
           '--epochs', '100', '--batchsize', '8',
           '--variants', *VARIANTS, '--seeds', *map(str, SEEDS)]
subprocess.run(command + ['--dry_run'], check=True)


In [ ]:
# Starts/resumes training. Run this cell again after a runtime reset (after setup).
subprocess.run(command, check=True)


In [ ]:
# Can also be run while only part of the study has finished.
subprocess.run([sys.executable, 'summarize_pid_v2_ablation.py', '--output_root', str(OUTPUT_ROOT),
                '--seeds', '42', '3407', '1234'], check=True)
from IPython.display import Markdown, display
display(Markdown((OUTPUT_ROOT / 'ablation_report' / 'summary.md').read_text()))


Giữ cả thư mục `pid_v2_ablation_v1` trên Drive: `ablation_study.json`, `snapshots`, `logs`, `results`, `ablation`, `ablation_report`.
`summary.csv` chứa cả 10 metrics và chênh lệch với v2 cùng seed; `dice_table.tex` dùng cho bảng paper.
Checkpoint chọn theo validation mDice; không chọn epoch/seed theo CVC-300. Đây là ablation của v2, chưa phải bằng chứng vượt EDANet.
